# 08 Build the report

Creates the Power BI report **WindFarm Operations Report** on the model: three pages, built from the measures.

1. **Site overview**: energy against budget by month, what energy was lost to, the headline figures.
2. **Turbine health**: one row per turbine, and the gearbox bearing temperatures over time.
3. **Money and market**: revenue against budget, repair cost, curtailment by reason.

If a report of that name exists, the notebook leaves it as it is and says so; delete the report first. The guide has the three pages
as a list of visuals, to build the report by hand if this notebook fails.

In [ ]:
%pip install -q semantic-link-labs==0.15.2

In [ ]:
import json
import uuid

MODEL = "WindFarm Operations"
REPORT = "WindFarm Operations Report"


def field(table, name, measure=False):
    return {"table": table, "name": name, "measure": measure}


def m(name):
    return field("KPIs", name, True)


def visual(kind, x, y, w, h, roles, title=None):
    """roles: {"Category": [fields], "Y": [fields], ...} in the order the visual shows them."""
    fields = [f for group in roles.values() for f in group]
    alias = {t: f"t{i}" for i, t in enumerate(dict.fromkeys(f["table"] for f in fields))}
    select = [{("Measure" if f["measure"] else "Column"): {"Expression": {"SourceRef": {"Source": alias[f["table"]]}},
                                                             "Property": f["name"]},
               "Name": f"{f['table']}.{f['name']}"} for f in fields]
    single = {"visualType": kind,
              "projections": {role: [{"queryRef": f"{f['table']}.{f['name']}"} for f in group]
                              for role, group in roles.items()},
              "prototypeQuery": {"Version": 2, "From": [{"Name": a, "Entity": t, "Type": 0} for t, a in alias.items()],
                                 "Select": select},
              "drillFilterOtherVisuals": True}
    if title:
        single["vcObjects"] = {"title": [{"properties": {
            "show": {"expr": {"Literal": {"Value": "true"}}},
            "text": {"expr": {"Literal": {"Value": "'" + title.replace("'", "''") + "'"}}}}}]}
    position = {"x": x, "y": y, "z": 0, "width": w, "height": h}
    config = {"name": uuid.uuid4().hex[:20], "layouts": [{"id": 0, "position": position}], "singleVisual": single}
    return {**position, "config": json.dumps(config), "filters": "[]"}


def cards(names, y=20):
    width = (1240 - 20 * (len(names) - 1)) / len(names)
    return [visual("card", 20 + i * (width + 20), y, width, 110, {"Values": [m(n)]}) for i, n in enumerate(names)]


def page(name, visuals):
    return {"name": uuid.uuid4().hex[:20], "displayName": name, "displayOption": 1, "width": 1280, "height": 720,
            "config": "{}", "filters": "[]", "visualContainers": visuals}


MONTH, YEAR, TURBINE = field("Date", "YearMonth"), field("Date", "Year"), field("Turbine", "TurbineId")
PAGES = [
    page("Site overview",
         cards(["Energy MWh", "Capacity Factor", "Time Availability", "Curtailment Share", "Revenue USD"])
         + [visual("slicer", 20, 150, 200, 240, {"Values": [YEAR]}, "Year"),
            visual("clusteredColumnChart", 240, 150, 620, 270, {"Category": [MONTH], "Y": [m("Energy MWh"), m("Budget P50 MWh")]},
                   "Energy and budget by month, MWh"),
            visual("clusteredBarChart", 880, 150, 380, 270, {"Category": [field("Scada", "LossCause")], "Y": [m("Lost Energy MWh")]},
                   "Lost energy by cause, MWh"),
            visual("lineChart", 20, 440, 620, 260, {"Category": [MONTH], "Y": [m("Time Availability"), m("Energy Availability")]},
                   "Availability by month"),
            visual("lineChart", 660, 440, 600, 260, {"Category": [MONTH], "Y": [m("Capacity Factor"), m("Wind Index")]},
                   "Capacity factor and wind index by month")]),
    page("Turbine health",
         [visual("slicer", 20, 20, 200, 300, {"Values": [YEAR]}, "Year"),
          visual("tableEx", 240, 20, 1020, 300,
                 {"Values": [TURBINE] + [m(n) for n in ["Energy MWh", "Capacity Factor", "Time Availability", "Performance Index",
                                                        "Failures", "MTTR Hours", "Average Gearbox Bearing Temp",
                                                        "Average Gearbox Vibration"]]}, "Turbines"),
          visual("lineChart", 20, 340, 620, 360, {"Category": [MONTH], "Series": [TURBINE], "Y": [m("Average Gearbox Bearing Temp")]},
                 "Gearbox bearing temperature by month, degC"),
          visual("lineChart", 660, 340, 600, 360, {"Category": [MONTH], "Series": [TURBINE], "Y": [m("Performance Index")]},
                 "Performance index by month")]),
    page("Money and market",
         cards(["Revenue USD", "Budget Revenue USD", "Repair Cost Paid by Owner USD", "Captured Price USD per MWh", "PM Compliance"])
         + [visual("slicer", 20, 150, 200, 240, {"Values": [YEAR]}, "Year"),
            visual("clusteredColumnChart", 240, 150, 1020, 270, {"Category": [MONTH], "Y": [m("Revenue USD"), m("Budget Revenue USD")]},
                   "Revenue and budget by month, USD"),
            visual("tableEx", 20, 440, 620, 260,
                   {"Values": [field("Curtailments", "Reason"), field("Curtailments", "Compensation"),
                               m("Deemed Generation MWh"), m("Unpaid Curtailment MWh")]}, "Curtailment by reason"),
            visual("lineChart", 660, 440, 600, 260,
                   {"Category": [MONTH], "Y": [m("Captured Price USD per MWh"), m("Mean Real-Time Price USD per MWh")]},
                   "Captured and mean real-time price by month, USD per MWh")]),
]
REPORT_JSON = {"config": json.dumps({"version": "5.37", "themeCollection": {}, "activeSectionIndex": 0}),
               "layoutOptimization": 0, "resourcePackages": [], "sections": PAGES}
print(len(PAGES), "pages,", sum(len(p["visualContainers"]) for p in PAGES), "visuals")

In [ ]:
from sempy_labs import report as rep

rep.create_report_from_reportjson(report=REPORT, dataset=MODEL, report_json=REPORT_JSON)
print("Report created:", REPORT, "- open it from the workspace")